# QueryChat - `on_tool_request` Experimentation

In this notebook I'll be experimenting with the `on_tool_request` command to experiment intercepting LLM tool calls to validate, log, or transform them before execution. 

## Experiment 1: Log tool calls

This experiment is mostly to check what tool the LLM requests and what arguments it sends. For this experiment I will print both the **tool name** and the **arguments**. This could be useful to serve as a baseline and to unedrstand how querychat behaves **before** actually changing anything.

In [37]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) # add project root to path

from dotenv import load_dotenv
from querychat import QueryChat
from chatlas import ChatAnthropic
import os
import pandas as pd
from src.data import load_dashboard_data
import re

In [12]:
# Experiment 1: Log tool requests with on_tool_request
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Load the same dataset used in the dashboard
df = load_dashboard_data()

# Reuse the same greeting and data description from app.py
ai_greeting = """
👋 Hi! I'm your AI burnout explorer.

Ask me questions about employee burnout, productivity, AI usage, workload, and work-life balance.

Examples:
- Show employees with high burnout risk
- Show employees with high AI usage and low productivity
- Sort employees by burnout risk from highest to lowest
- Which job roles have the highest burnout risk?
- Show employees with high manual work hours

You can also say Reset to clear the current AI filter/sort.
"""

ai_data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- focus_hours_per_day: average focus/deep work hours per day.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- burnout_risk_score: numeric burnout risk score.
- burnout_risk_level: burnout category label.
- productivity_score: numeric productivity score.
- work_life_balance_score: numeric work-life balance score.
- workload_score: derived workload metric combining manual work, meetings, and deadline pressure.
- workload_band: workload category (Low, Medium, High).
- ai_band: AI usage category (Low, Moderate, High).

This dataset can be used to analyze:
- Burnout risk by role or experience
- AI usage patterns across employees
- Links between productivity and burnout
- Work-life balance differences
- Manual work and deadline pressure patterns
- High-risk employee subgroups
"""

In [30]:
# Store logs here
tool_logs = []
current_prompt = None

# Callback for on_tool_request
def log_tool_request(req):
    global current_prompt

    print("\n--- TOOL REQUEST ---")
    print("Prompt:", current_prompt)
    print("Tool name:", req.name)
    print("Arguments:", req.arguments)

    tool_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": req.name,
            "arguments": str(req.arguments),
        }
    )

# Create QueryChat client
client = ChatAnthropic(model="claude-sonnet-4-0")

# Create QueryChat object
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

# Attach the callback to the client used by QueryChat
client = qc.client()
client.on_tool_request(log_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [ ]:
'''
test_prompts = [
    "Show employees with high burnout risk",
    "Show employees with high AI usage and low productivity",
    "Sort employees by burnout risk from highest to lowest",
    "Which job roles have the highest burnout risk?",
    "Show employees with high manual work hours"
]
'''
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
    "Group the dataset by job_role and compute average burnout_risk_score"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)

In [36]:
log_df = pd.DataFrame(tool_logs)
pd.set_option("display.max_colwidth", None)
log_df

,prompt,tool_name,arguments
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'Employees with burnout risk score > 80'}"
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}"
2,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'Employees with manual work > 40 hours, sorted by burnout risk (highest first)'}"
3,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': 'SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}"
4,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'Employees with burnout risk score > 80'}"
5,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}"
6,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'Employees with manual work > 40 hours, sorted by burnout risk (highest first)'}"
7,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': '-- Calculate average burnout risk score by job role for employees with manual work > 40 hours\nSELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nWHERE manual_work_hours_per_week > 40\nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role for filtered employees (manual work > 40 hours)'}"


### Experiment 1 - Results
This experiment confirms that `on_tool_request` successfully intercepts tool calls, different prompts trigger different tools, and that LLM translates the user's requests into SQL queries. **Filtering and sorting prompts** triggered `querychat_update_dashboard` since it applies SQL filters to the dataset. While **aggregation prompts** triggered the `querychat_query` since it runs SQL queries to compute summaries.

## Experiment 2: Validate tool calls

In this experiment I add simple rules to check whether the request looks acceptable before the execution. Some validation rules could be:
- only allow expected tools like `query` or `update_dashboard`
- reject empty arguments
- reject overly broad SQL queries like `SELECT *` without a filter
- reject requests that mention columns outside our dataset

For this experiment, I focus only on **validating column names** because it is simple, reliable, and directly related to the dataset schema. Since the dashboard relies on a fixed set of columns, checking whether the generated SQL references valid column names provides a lightweight way to detect potential errors in the LLM-generated query **without interfering with normal filtering or aggregation behavior**.

This experiment is so that `on_tool_request` improves the reliability of the LLM.

In [38]:
# Reject requests that mention columns outside our dataset

# Reset logs
validation_logs = []
current_prompt = None

# Get valid column names from the dataframe
valid_columns = set(df.columns)

def validate_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments
    query_text = arguments.get("query", "")

    # Find words that could be column names
    tokens = re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", query_text)

    # Keep only tokens that look like dataframe-style names
    candidate_columns = {token for token in tokens if "_" in token}

    # Check which are not real columns
    invalid_columns = sorted(candidate_columns - valid_columns)

    is_valid = len(invalid_columns) == 0

    print("\n--- VALIDATION CHECK ---")
    print("Prompt:", current_prompt)
    print("Tool name:", tool_name)
    print("Candidate columns:", sorted(candidate_columns))
    print("Invalid columns:", invalid_columns)
    print("Valid request:", is_valid)

    validation_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "arguments": str(arguments),
            "candidate_columns": str(sorted(candidate_columns)),
            "invalid_columns": str(invalid_columns),
            "is_valid": is_valid,
        }
    )

In [ ]:
client.on_tool_request(validate_tool_request)

# "Filter the dataset to employees with burnout_score > 80" is the "bad" prompt
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Group the dataset by job_role and compute average burnout_risk_score",
    "Filter the dataset to employees with burnout_score > 80"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)

In [46]:
validation_df = pd.DataFrame(validation_logs)
pd.set_option("display.max_colwidth", None)
validation_df

,prompt,tool_name,arguments,candidate_columns,invalid_columns,is_valid
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'Employees with burnout risk score > 80'}",['burnout_risk_score'],[],True
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with manual work hours > 40 per week'}",['manual_work_hours_per_week'],[],True
2,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': ""-- Calculate average burnout risk score by job role for employees with manual work > 40 hours\nSELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count,\n AVG(manual_work_hours_per_week) AS avg_manual_hours,\n AVG(deadline_pressure_level = 'High') AS pct_high_pressure\nFROM AIUsageBurnoutCheckup \nWHERE manual_work_hours_per_week > 40\nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC"", '_intent': 'Calculate average burnout risk score by job role for employees with high manual work hours'}","['avg_burnout_risk_score', 'avg_manual_hours', 'burnout_risk_score', 'deadline_pressure_level', 'employee_count', 'job_role', 'manual_work_hours_per_week', 'pct_high_pressure']","['avg_burnout_risk_score', 'avg_manual_hours', 'employee_count', 'pct_high_pressure']",False
3,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': ""-- Calculate average burnout risk score by job role for employees with manual work > 40 hours\nSELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count,\n AVG(manual_work_hours_per_week) AS avg_manual_hours,\n SUM(CASE WHEN deadline_pressure_level = 'High' THEN 1 ELSE 0 END) AS high_pressure_count\nFROM AIUsageBurnoutCheckup \nWHERE manual_work_hours_per_week > 40\nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC"", '_intent': 'Calculate average burnout risk score by job role for employees with high manual work hours'}","['avg_burnout_risk_score', 'avg_manual_hours', 'burnout_risk_score', 'deadline_pressure_level', 'employee_count', 'high_pressure_count', 'job_role', 'manual_work_hours_per_week']","['avg_burnout_risk_score', 'avg_manual_hours', 'employee_count', 'high_pressure_count']",False
4,Filter the dataset to employees with burnout_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 80', 'title': 'Employees with burnout risk score > 80'}",['burnout_risk_score'],[],True


### Experiment 2 - Results
This experiment demonstrates how `on_tool_request` can be used to inspect tool calls before execution. By extracting column-like tokens from the generated SQL and comparing them against the dataframe schema, we were able to detect whether the LLM referenced valid dataset columns.

The results show that simple filtering queries correctly referenced existing columns and passed validation. However, aggregation queries sometimes introduced derived column names (e.g., `avg_manual_hours`, `employee_count`) that were not part of the original dataset schema. These were flagged as invalid by the validation logic.

This experiment illustrates how `on_tool_request` can provide a lightweight validation layer that helps detect potential issues in automatically generated queries before they are executed.

## Experiment 3: Transform tool calls

In this experiment I will intercept the request to adjust it lightly before it runs. Some of the transformations I consider for this experiment are:
- cleaning the query title to standarize them
- normalize the column names
- improve query structure

This shows how `on_tool_request` can transform the tool requests before executions to something safer or more useful.